# 1 — Original LLM classifications

The recorded classifications are CSV inputs so they can be inspected and the paper's results reproduced without new model calls. Scores 1–5 correspond to negligible, low, medium, high, and catastrophic. Each experiment uses five runs of six models and batches of 20.

Inputs: `intermediate/s0_data.pkl`, `classification_descriptions.csv`, `classifications_original_randomized.csv`, and `classifications_original_fixed.csv`. Description IDs link labels to incident text; `fixed_position` records the original zero-based order within the fixed sample.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display


In [2]:
INTERMEDIATE = Path("intermediate")
prepared = pd.read_pickle(INTERMEDIATE / "s0_data.pkl")

claims = prepared["claims"].copy()
descriptions = pd.read_csv("classification_descriptions.csv").sort_values(
    "description_id"
).reset_index(drop=True)
description_ids = descriptions.set_index("Description").description_id
claims["description_id"] = claims.Description.map(description_ids)


In [3]:
MODEL_ORDER = [
    "GPT-5.6 Sol",
    "Gemini 3.1 Pro",
    "Kimi K2.6",
    "DeepSeek V4 Pro",
    "Qwen3.5 397B",
    "GLM-5.2",
]
LABEL_NAMES = ["negligible", "low", "medium", "high", "catastrophic"]
LABELS = LABEL_NAMES
RUNS = [1, 2, 3, 4, 5]
BATCH_SIZE = 20

labels = pd.read_csv("classifications_original_randomized.csv")
fixed_labels = pd.read_csv("classifications_original_fixed.csv")
original_ids = descriptions.description_id.tolist()
fixed_ids = (
    fixed_labels[["description_id", "fixed_position"]]
    .drop_duplicates()
    .sort_values("fixed_position")
    .description_id.tolist()
)
labels = labels.sort_values(["model", "run", "description_id"]).reset_index(drop=True)
fixed_labels = fixed_labels.drop(columns="fixed_position").sort_values(
    ["model", "run", "description_id"]
).reset_index(drop=True)


def label_cube(frame, ids):
    index = pd.MultiIndex.from_product(
        [ids, MODEL_ORDER, RUNS], names=["description_id", "model", "run"]
    )
    return (
        frame.set_index(["description_id", "model", "run"]).label_value
        .reindex(index).to_numpy().reshape(len(ids), len(MODEL_ORDER), len(RUNS))
    )


original_cube = label_cube(labels, original_ids)
fixed_cube = label_cube(fixed_labels, fixed_ids)
fixed_manifest = pd.DataFrame({"description_id": fixed_ids})
fixed_manifest = fixed_manifest.merge(descriptions, on="description_id", how="left")
fixed_manifest["fixed_position"] = np.arange(len(fixed_manifest))
fixed_manifest["batch_id"] = fixed_manifest.fixed_position // BATCH_SIZE
fixed_manifest["position_in_batch"] = fixed_manifest.fixed_position % BATCH_SIZE


## Model Settings and Prompt

OpenAI used no reasoning effort and no temperature. The other provides used temperature 0.1 with low Gemini thinking and disabled DeepInfra reasoning.

In [4]:
MODELS = [
    {
        "model": "GPT-5.6 Sol",
        "provider": "OpenAI",
        "model_id": "gpt-5.6-sol",
        "temperature": None,
        "reasoning": "none",
    },
    {
        "model": "Gemini 3.1 Pro",
        "provider": "Google",
        "model_id": "gemini-3.1-pro-preview",
        "temperature": 0.1,
        "reasoning": "low",
    },
    {
        "model": "Kimi K2.6",
        "provider": "DeepInfra",
        "model_id": "moonshotai/Kimi-K2.6",
        "temperature": 0.1,
        "reasoning": "none",
    },
    {
        "model": "DeepSeek V4 Pro",
        "provider": "DeepInfra",
        "model_id": "deepseek-ai/DeepSeek-V4-Pro",
        "temperature": 0.1,
        "reasoning": "none",
    },
    {
        "model": "Qwen3.5 397B",
        "provider": "DeepInfra",
        "model_id": "Qwen/Qwen3.5-397B-A17B",
        "temperature": 0.1,
        "reasoning": "none",
    },
    {
        "model": "GLM-5.2",
        "provider": "DeepInfra",
        "model_id": "zai-org/GLM-5.2",
        "temperature": 0.1,
        "reasoning": "none",
    },
]
roster = pd.DataFrame(MODELS)
display(roster[["model", "provider", "model_id", "temperature", "reasoning"]])


def make_prompt(batch):
    lines = "\n".join(f"{i + 1}. {text}" for i, text in enumerate(batch["Description"]))
    return (
        f"The following are descriptions of {len(batch)} property insurance incidents:\n{lines}\n"
        "You are an actuarial assistant assigned to classify these insurance incidents into one of five "
        "categories: negligible, low, medium, high, catastrophic. "
        f"Return a JSON object with a labels array containing exactly {len(batch)} classification names "
        "in the same order. Do not include explanations."
    )


def make_schema(n):
    return {
        "type": "object",
        "properties": {
            "labels": {
                "type": "array",
                "items": {"type": "string", "enum": LABELS},
                "minItems": n,
                "maxItems": n,
            }
        },
        "required": ["labels"],
        "additionalProperties": False,
    }

,model,provider,model_id,temperature,reasoning
0,GPT-5.6 Sol,OpenAI,gpt-5.6-sol,NaN,none
1,Gemini 3.1 Pro,Google,gemini-3.1-pro-preview,0.1,low
2,Kimi K2.6,DeepInfra,moonshotai/Kimi-K2.6,0.1,none
3,DeepSeek V4 Pro,DeepInfra,deepseek-ai/DeepSeek-V4-Pro,0.1,none
4,Qwen3.5 397B,DeepInfra,Qwen/Qwen3.5-397B-A17B,0.1,none
5,GLM-5.2,DeepInfra,zai-org/GLM-5.2,0.1,none


In [5]:
display(
    pd.DataFrame(
        {
            "experiment": ["Randomized batches", "Fixed batches"],
            "descriptions": [len(original_ids), len(fixed_ids)],
            "models": [len(MODEL_ORDER)] * 2,
            "runs": [len(RUNS)] * 2,
            "classifications": [len(labels), len(fixed_labels)],
        }
    )
)

,experiment,descriptions,models,runs,classifications
0,Randomized batches,4690,6,5,140700
1,Fixed batches,1000,6,5,30000


## Save results

In [6]:
fixed_prompts = {
    int(batch_id): make_prompt(batch)
    for batch_id, batch in fixed_manifest.groupby("batch_id")
}
pd.to_pickle(
    {
        "claims": claims,
        "descriptions": descriptions,
        "labels": labels,
        "fixed_labels": fixed_labels,
        "fixed_manifest": fixed_manifest,
        "original_cube": original_cube,
        "fixed_cube": fixed_cube,
        "original_ids": original_ids,
        "fixed_ids": fixed_ids,
        "model_order": MODEL_ORDER,
        "label_names": LABEL_NAMES,
        "runs": RUNS,
        "fixed_prompts": fixed_prompts,
    },
    INTERMEDIATE / "s1_labels.pkl",
)